In [ ]:
import numpy as np
from scipy.interpolate import interp1d
from scipy.stats import norm


# Market and model inputs used in the checks below.
S0 = 100.0
r = 0.03
q = 0.00
T = 1.0
K0 = 100.0

heston_params = {
    "kappa": 2.0,   # variance mean-reversion speed
    "theta": 0.04,  # long-run variance
    "sigma": 0.30,  # volatility of variance
    "rho": -0.60,   # spot/variance correlation
    "v0": 0.04,     # initial variance
}


def black_scholes_call(S0, K, T, r, q, sigma):
    """Closed-form Black-Scholes call price."""
    K = np.asarray(K, dtype=float)
    if T <= 0:
        return np.maximum(S0 - K, 0.0)
    vol_sqrt_T = sigma * np.sqrt(T)
    d1 = (np.log(S0 / K) + (r - q + 0.5 * sigma**2) * T) / vol_sqrt_T
    d2 = d1 - vol_sqrt_T
    return S0 * np.exp(-q * T) * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)


def heston_log_price_cf(u, S0, T, r, q, kappa, theta, sigma, rho, v0):
    """Characteristic function of log(S_T) under the risk-neutral Heston model."""
    u = np.asarray(u, dtype=complex)
    x0 = np.log(S0)

    # With zero vol-of-vol, variance is deterministic and the CF is Black-Scholes-like.
    if abs(sigma) < 1e-14:
        if abs(kappa) < 1e-14:
            int_var = v0 * T
        else:
            int_var = theta * T + (v0 - theta) * (1.0 - np.exp(-kappa * T)) / kappa
        drift = x0 + (r - q) * T - 0.5 * int_var
        return np.exp(1j * u * drift - 0.5 * u**2 * int_var)

    iu = 1j * u
    a = kappa * theta
    b = kappa - rho * sigma * iu
    d = np.sqrt(b**2 + sigma**2 * (u**2 + iu))
    g = (b - d) / (b + d)
    exp_dt = np.exp(-d * T)

    # Little Heston Trap form keeps the complex logarithm numerically stable.
    C = iu * (x0 + (r - q) * T) + (a / sigma**2) * ((b - d) * T - 2.0 * np.log((1.0 - g * exp_dt) / (1.0 - g)))
    D = ((b - d) / sigma**2) * ((1.0 - exp_dt) / (1.0 - g * exp_dt))
    return np.exp(C + D * v0)


def carr_madan_fft_call_grid(S0, T, r, q, params, alpha=1.5, N=2**12, eta=0.25):
    """Price calls on a log-strike grid using the Carr-Madan damped FFT."""
    j = np.arange(N)
    v = eta * j
    lam = 2.0 * np.pi / (N * eta)
    b = 0.5 * N * lam
    k = -b + lam * j

    # Simpson weights improve the integral accuracy with the same FFT size.
    weights = np.ones(N)
    weights[0] = 1.0 / 3.0
    weights[1::2] = 4.0 / 3.0
    weights[2::2] = 2.0 / 3.0

    u = v - 1j * (alpha + 1.0)
    phi = heston_log_price_cf(u, S0, T, r, q, **params)
    denominator = alpha**2 + alpha - v**2 + 1j * (2.0 * alpha + 1.0) * v
    psi = np.exp(-r * T) * phi / denominator

    # FFT evaluates the damped Fourier integral at all log-strikes at once.
    fft_input = np.exp(1j * b * v) * psi * eta * weights
    fft_values = np.fft.fft(fft_input).real
    strikes = np.exp(k)
    calls = np.exp(-alpha * k) * fft_values / np.pi

    valid = np.isfinite(calls) & (calls > 0.0)
    return strikes[valid], calls[valid]


def carr_madan_fft_call(strikes, S0, T, r, q, params, alpha=1.5, N=2**12, eta=0.25):
    """Interpolate Carr-Madan FFT prices from the computed strike grid."""
    grid_K, grid_C = carr_madan_fft_call_grid(S0, T, r, q, params, alpha=alpha, N=N, eta=eta)
    f = interp1d(grid_K, grid_C, kind="cubic", bounds_error=False, fill_value="extrapolate")
    return np.asarray(f(strikes), dtype=float)


def heston_mc_call(S0, K, T, r, q, params, n_paths=120_000, n_steps=252, seed=7):
    """Monte Carlo Heston call using full-truncation Euler and antithetic normals."""
    rng = np.random.default_rng(seed)
    half_paths = n_paths // 2
    dt = T / n_steps
    sqrt_dt = np.sqrt(dt)

    log_S = np.full(n_paths, np.log(S0), dtype=float)
    v = np.full(n_paths, params["v0"], dtype=float)

    for _ in range(n_steps):
        z1_half = rng.standard_normal(half_paths)
        z2_half = rng.standard_normal(half_paths)
        z1 = np.concatenate([z1_half, -z1_half])
        z2 = np.concatenate([z2_half, -z2_half])
        zv = params["rho"] * z1 + np.sqrt(1.0 - params["rho"]**2) * z2

        v_pos = np.maximum(v, 0.0)
        log_S += (r - q - 0.5 * v_pos) * dt + np.sqrt(v_pos) * sqrt_dt * z1
        v += params["kappa"] * (params["theta"] - v_pos) * dt + params["sigma"] * np.sqrt(v_pos) * sqrt_dt * zv

    payoff = np.maximum(np.exp(log_S) - K, 0.0)
    discounted = np.exp(-r * T) * payoff
    price = discounted.mean()
    stderr = discounted.std(ddof=1) / np.sqrt(n_paths)
    return price, stderr


# Check 1: zero vol-of-vol with v0=theta reduces exactly to Black-Scholes.
bs_sigma = 0.20
bs_limit_params = {"kappa": 1.0, "theta": bs_sigma**2, "sigma": 0.0, "rho": 0.0, "v0": bs_sigma**2}
test_strikes = np.array([70, 80, 90, 100, 110, 120, 130], dtype=float)
fft_bs = carr_madan_fft_call(test_strikes, S0, T, r, q, bs_limit_params, N=2**13, eta=0.15)
closed_bs = black_scholes_call(S0, test_strikes, T, r, q, bs_sigma)
bs_errors = np.abs(fft_bs - closed_bs)

print("Black-Scholes collapse check")
for K, fft_price, bs_price, err in zip(test_strikes, fft_bs, closed_bs, bs_errors):
    print(f"K={K:6.1f}  FFT={fft_price:10.6f}  BS={bs_price:10.6f}  abs_err={err:.2e}")
print(f"max_abs_err={bs_errors.max():.2e}\n")

# Check 2: compare one Heston FFT price with a Monte Carlo confidence interval.
fft_heston = carr_madan_fft_call(np.array([K0]), S0, T, r, q, heston_params, N=2**13, eta=0.15)[0]
mc_heston, mc_se = heston_mc_call(S0, K0, T, r, q, heston_params)
mc_low, mc_high = mc_heston - 1.96 * mc_se, mc_heston + 1.96 * mc_se

print("Heston FFT vs Monte Carlo check")
print(f"FFT price       = {fft_heston:.6f}")
print(f"MC price        = {mc_heston:.6f}")
print(f"MC 95% interval = [{mc_low:.6f}, {mc_high:.6f}]")
print(f"abs_diff        = {abs(fft_heston - mc_heston):.6f}")

assert bs_errors.max() < 1e-3, "FFT should collapse to Black-Scholes in the zero vol-of-vol limit."
assert mc_low <= fft_heston <= mc_high, "FFT price should fall inside the Monte Carlo 95% interval."
